In [1]:
#准备文档

from langchain_text_splitters import MarkdownHeaderTextSplitter

def prepare_docs():
    # =============1.准备文档================
    with open('./resources/评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    return docs

docs = prepare_docs()

In [2]:
#向量模型
from langchain_ollama import OllamaEmbeddings

# 1.初始化向量模型，可以把文档转稠密向量
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

In [9]:
#4.collection操作
# 最终我们向Collection中存入的数据结构应该是这样的：
# {
#     "id": 1,
#     "h2": "存h2的内容作为示例",
#     "content": "存page_content，也就是文档内容",
#     "dense": [0.1, ...], # 存稠密向量，也就是浮点数数组，值根据content用向量模型生成
#     "sparse": [0.2, ...] # 存稀疏向量，也就是浮点数数组，值根据content用BM25生成
# }

from pymilvus import MilvusClient

# 初始化客户端
client = MilvusClient(uri="http://127.0.0.1:19530")

# Collection名称
collection_name = "my_collection_1"

In [10]:
from pymilvus import Function, FunctionType, DataType

def create_collection():
    # 1.schema
    schema = client.create_schema(
        auto_id=False,  # 不自动生成id
        enable_dynamic_field=False,  # 是否允许动态字段
    )

    # 1.1.主键字段
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    # 1.2.稠密向量字段
    schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024)
    # 1.3.稀疏向量字段
    schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
    # 1.4.普通元数据字段，用来做数据过滤
    schema.add_field(field_name="h2", datatype=DataType.VARCHAR, max_length=128)
    # 1.5.原始文档内容，需要被向量化
    schema.add_field(
        field_name="content",
        datatype=DataType.VARCHAR,
        max_length=2048,
        enable_analyzer=True,  # 允许对该字段分词，转BM25稀疏向量
        analyzer_params={"type": "chinese"}  # 指定分词方式，这里选择中文分词
    )
    # 1.6.添加BM25函数，可以自动把原始文档转BM25向量存储
    bm25_function = Function(
        name="text_bm25_emb",  # Function name
        input_field_names=["content"],  # 包含原始文档数据的字段的名称
        output_field_names=["sparse"],  # 稀疏向量字段的名称
        function_type=FunctionType.BM25,  # Set to `BM25`
    )
    schema.add_function(bm25_function)
    # 1.7.添加Embedding函数，可以自动把原始文档转稠密向量存储
    text_embedding_function = Function(
        name="ali",                                     # Embedding函数的名称
        function_type=FunctionType.TEXTEMBEDDING,       # 函数类型，选择embedding function
        input_field_names=["content"],                  # 用来转稠密向量的原始文本字段的名称
        output_field_names=["dense"],                   # 存储稠密向量字段的名称
        params={                                        # 向量模型提供者的相关参数
            "provider": "dashscope",                    # 模型提供者名称，例如"dashscope"
            "model_name": "text-embedding-v3",          # 向量模型名称
            "dim": "1024",                              # 其它可选字段: 例如向量维度
        }
    )
    # 将配置好的Embedding函数添加到现有的Schema中
    schema.add_function(text_embedding_function)
    # 2.索引
    index_params = client.prepare_index_params()
    # 2.1.稠密向量索引
    index_params.add_index(
        field_name="dense",  # 字段名
        index_type="AUTOINDEX",  # 稠密向量索引方式
        metric_type="COSINE"  # 相似度计算方式，COSINE、IP
    )
    # 2.2.稀疏向量索引
    index_params.add_index(
        field_name="sparse",  # 字段名
        index_type="SPARSE_INVERTED_INDEX",  # 稀疏向量，采用倒排索引
        metric_type="BM25",  # 相似度计算方式，支持 BM25、IP
    )
    # 2.3.普通字段索引
    index_params.add_index(field_name="h2", index_type="AUTOINDEX")

    # 3.创建collection
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params
    )

create_collection()

In [11]:
# 查看Collection
res = client.list_collections()

print(res)  # ['my_collection_1']

['test', 'my_collection_1']


In [14]:
#5.insert操作

# milvus接收的是dict，所以先做数据转换
data = [
    {
        "id": i+1,
        "content": doc.page_content,
        "h2": doc.metadata.get("h2", ""),
        # "dense": vectors[i],  # 稠密向量字段，如果配置了embedding_function，就无需手动指定
    }
    for i, doc in enumerate(docs)
]

In [13]:
# 插入
res = client.insert(collection_name, data)
print(res)

#{'insert_count': 21, 'ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]}

{'insert_count': 21, 'ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]}


In [15]:
# 5.2 upsert
# upsert会先根据id删除旧的，再新增一条数据：
# - 如果id存在，等同于更新
# - 如果id不存在，等同于新增

# 1.准备两条数据

# id为1的数据，修改其h2字段值，模拟更新
data1 = data[0]
data1['h2'] = "测试一下"

 # id为2的数据，将其id修改为88，模拟新增
data2 = data[1]
data2['id'] = 88

# 2.测试upsert
res = client.upsert(collection_name, [data1, data2])
print(res)# {'upsert_count': 2, 'ids': [1, 88]}

{'upsert_count': 2, 'ids': [1, 88]}


In [16]:
# 根据id删除
res = client.delete(collection_name, ids=[88])
print(res) # {'delete_count': 1, 'cost': 0}

{'delete_count': 1}


In [ ]:
# 定义一个稠密检索的函数，擅长语义检索不擅长关键字检索
def dense_search(query: str):
    # 1.先把问题向量化
    query_vector = embeddings.embed_query(query)

    # 2.然后搜索：
    res = client.search(
        collection_name=collection_name,    # Collection名称
        anns_field="dense",                 # 检索字段名（稠密）
        data=[query_vector],                # 问题向量（稠密）, 也可以是问题本身（有向量函数时）
        limit=3,                            # 查询文档最大数量
        output_fields=["content", "h2"]  # 要返回的字段名，默认只返回id和score
    )
    # 3.遍历结果
    for hits in res:
        for hit in hits:
            print(hit)


dense_search("教育的负面作用是什么")

{'id': 14, 'distance': -0.00031384170870296657, 'entity': {'content': '### （四）古代学校教育特点  \n古代产生了学校，教育成为社会的专门职能，教学组织形式是“个别教育”。  \n古代学校教育与生产劳动相脱离，具有非生产性。  \n古代学校教育适应古代经济政治发展，有阶级性，封建社会的学校还有等级性。  \n古代学校教育适应古代思想文化发展，有道统性、专制性、刻板性、象征性。  \n古代学校教育初步发展，没有形成复杂的结构体系。  \n中国奴隶社会教学内容：六艺。  \n封建社会教育内容：四书五经。  \n西欧封建社会、教会学校教育内容：七艺。  \n骑士学校教育内容：七技。', 'h2': '第三节 教育的起源和发展'}}
{'id': 13, 'distance': -0.007080842275172472, 'entity': {'content': '### （三）学校教育产生的条件  \n生产力的发展以及社会生产水平的提高为学校产生提供了物质基础。  \n脑力劳动和体力劳动相分离为学校提供了专门从事教育活动的知识分子。  \n文字创造、知识的积累为学校教育活动的开展提供了有效的教育手段与充分的教育内容。  \n国家机器的产生需要专门的机构来培养官吏和知识分子来为统治阶级服务。', 'h2': '第三节 教育的起源和发展'}}
{'id': 16, 'distance': -0.011075947433710098, 'entity': {'content': '## 第二节 教育和社会的关系  \n### （一）政治与教育的关系  \n政治（经济制度）对教育具有决定作用，是决定教育性质的直接因素。具体表现为：决定了教育的性质和目的；决定了教育的领导权；决定了哪部分社会成员享有受教育的权利；决定了部分的教育内容；决定了教育的管理体制。  \n教育对政治发展的作用表现在：教育促进人的政治社会化；教育培养现代政治法律人才；教育促进现代政治民主化。', 'h2': '第二节 教育和社会的关系'}}


In [18]:
# 定义稀疏检索函数，擅长关键字检索不擅长语义检索
def sparse_search(query: str):
    # 1.直接检索，内置的hm25函数会自动把query转向量
    res = client.search(
        collection_name=collection_name,  # Collection名称
        data=[query],
        anns_field="sparse",  # 检索字段名（稀疏）
        limit=3,  # 查询文档最大数量
        output_fields=["content", "h2"]  # 要返回的字段名
    )
    # 2.打印结果
    for hits in res:
        for hit in hits:
            print(hit)


# 测试模糊的语义搜索，稀疏检索不太擅长
sparse_search("教育的负面作用是什么")

{'id': 21, 'distance': 1.9043395519256592, 'entity': {'h2': '第二节 教育和社会的关系', 'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。'}}
{'id': 20, 'distance': 1.871191143989563, 'entity': {'h2': '第二节 教育和社会的关系', 'content': '### （五）人口与教育的关系  \n人口对教育的作用主要表现在：人口的数量和增长是决定教育事业的规模、速度的一个重要因素；人口的质量影响教育的质量；人口的结构影响着教育的发展；人口流动影响着教育的发展。  \n教育对人口的作用主要表现在：控制人口的数量；提高人口的质量；改善人口结构；调整人才构成与流动等作用。'}}
{'id': 5, 'distance': 1.8368871212005615, 'entity': {'h2': '第一节 中外教育家及其教育思想', 'content': '### （五）荀子主要思想  \n**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  \n**教学原则**：学以致用，锲而不舍。'}}


In [20]:
#通用混合检索函数
# 加权融合策略 混合检索
from pymilvus import AnnSearchRequest

def hybrid_search(query: str, ranker):
    # 1.创建稠密、稀疏请求
    # 稠密
    dense_request = AnnSearchRequest(
        data=[query],
        anns_field="dense",
        param={},
        limit=3,
    )
    # 稀疏
    sparse_request = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={},
        limit=3,
    )
    # 混合在一起
    reqs = [dense_request, sparse_request]

    # 2.设定融合策略，发送请求
    res = client.hybrid_search(
        collection_name=collection_name,
        reqs=reqs,  #混合检索的请求
        ranker=ranker, #融合策略
        limit=2,
        output_fields=['content', 'h2']
    )
    for hits in res:
        print("TopK results:")
        for hit in hits:
            print(hit)


from pymilvus import WeightedRanker

# 定义加权融合策略
ranker = WeightedRanker(0.6, 0.4)
# 测试
hybrid_search("教育的负面作用是什么", ranker)

TopK results:
{'id': 21, 'distance': 0.759361743927002, 'entity': {'h2': '第二节 教育和社会的关系', 'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。'}}
{'id': 15, 'distance': 0.5011395215988159, 'entity': {'h2': '第一节 教育的功能', 'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。'}}


In [21]:
#RRF 倒数排名融合 是一种用于融合多个检索结果列表的算法，它与多路召回的文档得分无关，只关心排名
from pymilvus import AnnSearchRequest, RRFRanker

# 定义RRF融合策略
ranker = RRFRanker(k=60)
# 测试
hybrid_search("教育的负面作用是什么", ranker)

TopK results:
{'id': 21, 'distance': 0.032522473484277725, 'entity': {'content': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。', 'h2': '第二节 教育和社会的关系'}}
{'id': 15, 'distance': 0.016393441706895828, 'entity': {'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h2': '第一节 教育的功能'}}


In [ ]:
# Cross-Encoder，交叉编码，把用户问题和每个文档一起输入模型，精确打分。

# cross-encoder rerank
from pymilvus import Function, FunctionType


def create_cross_encoder_ranker(query: str):
    return Function(
        name="dashscope_semantic_ranker",      # ranker名词，唯一即可
        input_field_names=["content"],          # 原始文档字段
        function_type=FunctionType.RERANK,  # ranker类型，这里是固定值
        params={
            "reranker": "model",  # rerank类型，基于模型rerank，也就是cross-encoder
            "provider": "ali",  # rerank模型提供者
            "model_name": "gte-rerank-v2",  # rerank模型名称
            "queries": [query],  # 查询条件
            "max_client_batch_size": 5,  # 单批次中处理的文档最大数量
        },
    )

def cross_encoder_hybrid_search(query: str):
    # 初始化一个cross-encoder的reranker
    cross_encoder_ranker = create_cross_encoder_ranker(query)
    # 混合检索
    hybrid_search(query, cross_encoder_ranker)
    

cross_encoder_hybrid_search("教育的负面作用是什么")

TopK results:
{'id': 15, 'distance': 0.29470494389533997, 'entity': {'h2': '第一节 教育的功能', 'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。'}}
{'id': 16, 'distance': 0.1490025371313095, 'entity': {'h2': '第二节 教育和社会的关系', 'content': '## 第二节 教育和社会的关系  \n### （一）政治与教育的关系  \n政治（经济制度）对教育具有决定作用，是决定教育性质的直接因素。具体表现为：决定了教育的性质和目的；决定了教育的领导权；决定了哪部分社会成员享有受教育的权利；决定了部分的教育内容；决定了教育的管理体制。  \n教育对政治发展的作用表现在：教育促进人的政治社会化；教育培养现代政治法律人才；教育促进现代政治民主化。'}}


: 